In [2]:
import ROOT

In [1]:
result_file = "/d/home/septian/Moments2Amplitudes/macros/result_tree_L4_only_0_nominal_t010030.root"

In [3]:
f = ROOT.TFile.Open(result_file, "READ")
f.ls()

TFile**		/d/home/septian/Moments2Amplitudes/macros/result_tree_L4_only_0_nominal_t010030.root	
 TFile*		/d/home/septian/Moments2Amplitudes/macros/result_tree_L4_only_0_nominal_t010030.root	
  KEY: TTree	result;1	result
  KEY: TTree	mass_dependent_params;1	Mass Dependent Parameters
  KEY: TTree	chi2;1	chi2


In [5]:
t_result = f.Get("result")
t_result.Print()

******************************************************************************
*Tree    :result    : result                                                 *
*Entries :       20 : Total =           85211 bytes  File  Size =      20650 *
*        :          : Tree compression factor =   1.68                       *
******************************************************************************
*Br    0 :mass_bin  : mass_bin/D                                             *
*Entries :       20 : Total  Size=        742 bytes  File Size  =        206 *
*Baskets :        1 : Basket Size=      32000 bytes  Compression=   1.15     *
*............................................................................*
*Br    1 :a_0_0     : a_0_0/D                                                *
*Entries :       20 : Total  Size=        727 bytes  File Size  =         95 *
*Baskets :        1 : Basket Size=      32000 bytes  Compression=   2.46     *
*...................................................

In [21]:
df_result = ROOT.RDataFrame(t_result)
df_result = df_result.Define("a_2_2_sqr", "a_2_2*a_2_2")

In [22]:
hist_a_2_2_sqr = df_result.Histo1D("mass_bin","a_2_2_sqr")

In [ ]:
canvas = ROOT.TCanvas("canvas","canvas",800,600)
hist_a_2_2_sqr.Draw("HIST C")
canvas.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas


In [6]:
t_mdparams = f.Get("mass_dependent_params")
t_mdparams.Print()

******************************************************************************
*Tree    :mass_dependent_params: Mass Dependent Parameters                              *
*Entries :        1 : Total =            7028 bytes  File  Size =       1987 *
*        :          : Tree compression factor =   1.00                       *
******************************************************************************
*Br    0 :MD_a_2_-1_a2_1320_k : MD_a_2_-1_a2_1320_k/D                        *
*Entries :        1 : Total  Size=        660 bytes  File Size  =        111 *
*Baskets :        1 : Basket Size=      32000 bytes  Compression=   1.00     *
*............................................................................*
*Br    1 :MD_a_2_-2_a2_1320_k : MD_a_2_-2_a2_1320_k/D                        *
*Entries :        1 : Total  Size=        660 bytes  File Size  =        111 *
*Baskets :        1 : Basket Size=      32000 bytes  Compression=   1.00     *
*........................................

In [33]:
t_mdparams.GetEntry(0)  # Load entry 0 (first entry)
a_2_2_coupling = t_mdparams.GetLeaf("MD_a_2_2_a2_1320_k").GetValue()
print("a_2_2 coupling for a2(1320): ", a_2_2_coupling)

a_2_2 coupling for a2(1320):  -27.153505703175234


In [37]:
mass_bins = [float(bin_center)/1000 for bin_center in range(820, 1580, 20)]  # Example mass bins from 820 to 1580 MeV with 20 MeV width, converted to GeV

In [39]:
def GetBWMagnitude(mass, coupling):
    mass0 = 1.3182
    width = 0.107
    denominator = complex(mass0**2 - mass**2, -mass0*width)
    magnitude = abs(coupling / denominator)
    return magnitude

In [50]:
a_2_vals = []
for i in mass_bins:
    a_2_vals.append(pow(GetBWMagnitude(i, a_2_2_coupling),2))

In [44]:
import numpy as np

In [51]:
gr_a_2_2 = ROOT.TGraph(len(mass_bins), np.array(mass_bins, dtype='float64'), np.array(a_2_vals, dtype='float64'))
gr_a_2_2.SetLineColor(ROOT.kRed)
gr_a_2_2.SetLineWidth(2)
gr_a_2_2.SetTitle("a_2_2 Magnitude from BW Model;Mass (GeV);|a_2_2|")


In [53]:
canvas.Clear()
hist_a_2_2_sqr.Draw("HIST C")
gr_a_2_2.Draw("C same")
canvas.Draw()